<a href="https://colab.research.google.com/github/KarenInfanta/openfda-adverse-event-analysis/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pharmaceutical Adverse Event Analysis Using FDA Public Data

## Business problem
Healthcare and pharmaceutical teams review large volumes of spontaneous
adverse-event reports. This project shows how reported events can be
organized and summarized descriptively, and what the data can and cannot say.

## Research questions
1. How do report counts vary over time?
2. Which drugs appear most often in reports?
3. Which reactions are reported most often?
4. What age and sex distributions are recorded?
5. What outcomes are recorded most often?
6. How do reported reactions vary by drug?
7. What data-quality issues could affect these findings?

## Scope
In scope: descriptive analysis of a January 2024 sample from the openFDA
drug adverse-event API (FAERS data).
Out of scope: causality, incidence or risk rates, ranking drugs by danger,
clinical recommendations.

## Key limitations
- Reports do not establish that a drug caused a reaction.
- Reports may be incomplete or duplicated.
- Counts cannot be used to calculate how common an event is.
- The sample is stratified by day (up to 100 reports/day), not random.

In [ ]:
import json
import requests

BASE_URL = "https://api.fda.gov/drug/event.json"

params = {
    "search": "receivedate:[20240101 TO 20240131]",
    "limit": 3,
}

# 1. Make the request, with basic failure handling
try:
    response = requests.get(BASE_URL, params=params, timeout=30)
    print("HTTP status:", response.status_code)
    print("Rate-limit headers:", {
        k: v for k, v in response.headers.items() if "ratelimit" in k.lower()
    })
    response.raise_for_status()
except requests.exceptions.HTTPError:
    # openFDA returns 404 when a query matches nothing
    print("Request returned an error:", response.text[:500])
    raise
except requests.exceptions.RequestException as e:
    print("Network problem:", e)
    raise

payload = response.json()

# 2. Inspect the metadata
meta = payload.get("meta", {})
print("\nLast updated by FDA:", meta.get("last_updated"))
print("Total matching reports:", meta.get("results", {}).get("total"))
print("\nDisclaimer:\n", meta.get("disclaimer"))

# 3. Summarize each record, without assuming any field exists
results = payload.get("results", [])
print(f"\nRecords returned: {len(results)}")

for rec in results:
    patient = rec.get("patient", {})
    drugs = patient.get("drug", [])
    reactions = patient.get("reaction", [])
    print({
        "safetyreportid": rec.get("safetyreportid"),
        "receivedate": rec.get("receivedate"),
        "patientsex": patient.get("patientsex"),
        "patientonsetage": patient.get("patientonsetage"),
        "n_drugs": len(drugs),
        "n_reactions": len(reactions),
        "serious": rec.get("serious"),
    })

# 4. Look at the raw structure of ONE record (truncated)
if results:
    print("\nFirst record (truncated):")
    print(json.dumps(results[0], indent=2)[:3000])

HTTP status: 200
Rate-limit headers: {}

Last updated by FDA: 2026-07-30
Total matching reports: 106399

Disclaimer:
 Do not rely on openFDA to make decisions regarding medical care. While we make every effort to ensure that data is accurate, you should assume all results are unvalidated. We may limit or otherwise restrict your access to the API in line with our Terms of Service.

Records returned: 3
{'safetyreportid': '23353582', 'receivedate': '20240101', 'patientsex': '2', 'patientonsetage': '50', 'n_drugs': 8, 'n_reactions': 1, 'serious': '2'}
{'safetyreportid': '23353583', 'receivedate': '20240101', 'patientsex': '1', 'patientonsetage': '89', 'n_drugs': 1, 'n_reactions': 3, 'serious': '2'}
{'safetyreportid': '23353584', 'receivedate': '20240101', 'patientsex': '1', 'patientonsetage': None, 'n_drugs': 1, 'n_reactions': 1, 'serious': '2'}

First record (truncated):
{
  "safetyreportversion": "1",
  "safetyreportid": "23353582",
  "primarysourcecountry": "US",
  "occurcountry": "US",

In [ ]:
import json
import time
from datetime import date, timedelta, datetime, timezone
from pathlib import Path

import requests

BASE_URL = "https://api.fda.gov/drug/event.json"
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
MAX_SKIP = 25_000  # openFDA's documented skip ceiling (approx.)


def api_get(params, max_retries=5, pause=0.3):
    """GET with retries and backoff.
    Returns parsed JSON, or None if the query matches nothing (HTTP 404)."""
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(BASE_URL, params=params, timeout=30)
        except requests.exceptions.RequestException as e:
            wait = 2 ** attempt
            print(f"  network error ({e}); retry {attempt}/{max_retries} in {wait}s")
            time.sleep(wait)
            continue

        if r.status_code == 200:
            time.sleep(pause)  # stay well under the rate limit
            return r.json()
        if r.status_code == 404:  # openFDA's "no matches found"
            return None
        if r.status_code in (429, 500, 502, 503, 504):
            wait = 2 ** attempt
            print(f"  HTTP {r.status_code}; retry {attempt}/{max_retries} in {wait}s")
            time.sleep(wait)
            continue
        r.raise_for_status()  # e.g. 400 = malformed query -> fail loudly
    raise RuntimeError(f"Gave up after {max_retries} retries: {params}")


def fetch_window(start, end, max_records=100, page_size=100):
    """Fetch up to max_records reports received between start and end (YYYYMMDD).
    Paginates with skip/limit. Returns dict with records, total, last_updated."""
    search = f"receivedate:[{start} TO {end}]"
    records, skip, total, last_updated = [], 0, 0, None

    while len(records) < max_records and skip <= MAX_SKIP:
        limit = min(page_size, max_records - len(records))
        payload = api_get({"search": search, "limit": limit, "skip": skip})
        if payload is None:  # empty result
            break
        meta = payload.get("meta", {})
        total = meta.get("results", {}).get("total", total)
        last_updated = meta.get("last_updated", last_updated)
        batch = payload.get("results", [])
        if not batch:
            break
        records.extend(batch)
        skip += len(batch)
        if skip >= total:
            break

    return {"records": records, "total": total, "last_updated": last_updated}

In [ ]:
START, END = date(2024, 1, 1), date(2024, 1, 31)
PER_DAY_CAP = 100

all_records, day_log, last_updated = [], [], None
d = START
while d <= END:
    ds = d.strftime("%Y%m%d")
    out = fetch_window(ds, ds, max_records=PER_DAY_CAP)
    all_records.extend(out["records"])
    last_updated = out["last_updated"] or last_updated
    day_log.append({"date": ds, "available": out["total"], "fetched": len(out["records"])})
    d += timedelta(days=1)

# True daily report counts (server-side aggregation, nothing downloaded)
count_payload = api_get({
    "search": f"receivedate:[{START:%Y%m%d} TO {END:%Y%m%d}]",
    "count": "receivedate",
})
daily_counts = count_payload["results"] if count_payload else []

# Save raw data together with provenance
raw_file = RAW_DIR / "drug_event_sample_2024-01.json"
with open(raw_file, "w") as f:
    json.dump({
        "provenance": {
            "endpoint": BASE_URL,
            "window": [START.isoformat(), END.isoformat()],
            "per_day_cap": PER_DAY_CAP,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "openfda_last_updated": last_updated,
        },
        "day_log": day_log,
        "daily_counts": daily_counts,
        "records": all_records,
    }, f)

# ---- Sanity checks ----
ids = [r.get("safetyreportid") for r in all_records]
print(f"Records fetched:          {len(all_records):,}")
print(f"Unique safetyreportid:    {len(set(ids)):,}")
print(f"Days with data:           {sum(1 for x in day_log if x['fetched'] > 0)}/{len(day_log)}")
print(f"Sum of daily counts:      {sum(x['count'] for x in daily_counts):,}  (compare to 106,399)")
print(f"Missing patient age:      {sum(1 for r in all_records if not r.get('patient', {}).get('patientonsetage')):,}")
print(f"Missing patient sex:      {sum(1 for r in all_records if not r.get('patient', {}).get('patientsex')):,}")
print(f"File size:                {raw_file.stat().st_size / 1e6:.1f} MB")

Records fetched:          3,100
Unique safetyreportid:    3,100
Days with data:           31/31
Sum of daily counts:      106,399  (compare to 106,399)
Missing patient age:      1,276
Missing patient sex:      441
File size:                73.2 MB


In [ ]:
import json
from pathlib import Path
import pandas as pd

RAW_FILE = Path("data/raw/drug_event_sample_2024-01.json")
PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

with open(RAW_FILE) as f:
    raw = json.load(f)
records = raw["records"]
print("Loaded records:", len(records))


def join_list(x):
    """openFDA often stores values as lists; join them, or return None."""
    return "|".join(x) if isinstance(x, list) and x else None


report_rows, drug_rows, reaction_rows = [], [], []

for rec in records:
    rid = rec.get("safetyreportid")
    patient = rec.get("patient") or {}
    drugs = patient.get("drug") or []
    reactions = patient.get("reaction") or []

    report_rows.append({
        "safetyreportid": rid,
        "receivedate": rec.get("receivedate"),
        "receiptdate": rec.get("receiptdate"),
        "reporttype": rec.get("reporttype"),
        "serious": rec.get("serious"),
        "seriousnessdeath": rec.get("seriousnessdeath"),
        "seriousnesslifethreatening": rec.get("seriousnesslifethreatening"),
        "seriousnesshospitalization": rec.get("seriousnesshospitalization"),
        "seriousnessdisabling": rec.get("seriousnessdisabling"),
        "seriousnesscongenitalanomali": rec.get("seriousnesscongenitalanomali"),
        "seriousnessother": rec.get("seriousnessother"),
        "patient_age_raw": patient.get("patientonsetage"),
        "patient_age_unit": patient.get("patientonsetageunit"),
        "patient_agegroup": patient.get("patientagegroup"),
        "patient_sex": patient.get("patientsex"),
        "primarysourcecountry": rec.get("primarysourcecountry"),
        "duplicate_flag": rec.get("duplicate"),
        "companynumb": rec.get("companynumb"),
        "n_drugs": len(drugs),
        "n_reactions": len(reactions),
    })

    for i, d in enumerate(drugs, start=1):
        openfda = d.get("openfda") or {}
        drug_rows.append({
            "safetyreportid": rid,
            "drug_seq": i,
            "medicinalproduct": d.get("medicinalproduct"),
            "drugcharacterization": d.get("drugcharacterization"),
            "brand_name": join_list(openfda.get("brand_name")),
            "generic_name": join_list(openfda.get("generic_name")),
            "activesubstancename": (d.get("activesubstance") or {}).get("activesubstancename"),
            "drugindication": d.get("drugindication"),
        })

    for j, r in enumerate(reactions, start=1):
        reaction_rows.append({
            "safetyreportid": rid,
            "reaction_seq": j,
            "reactionmeddrapt": r.get("reactionmeddrapt"),
            "reactionoutcome": r.get("reactionoutcome"),
        })

reports = pd.DataFrame(report_rows)
drugs = pd.DataFrame(drug_rows)
reactions = pd.DataFrame(reaction_rows)

reports.to_csv(PROC_DIR / "reports_extracted.csv", index=False)
drugs.to_csv(PROC_DIR / "drugs_extracted.csv", index=False)
reactions.to_csv(PROC_DIR / "reactions_extracted.csv", index=False)

# ---- Quality checks ----
print("\nShapes -> reports:", reports.shape, "| drugs:", drugs.shape, "| reactions:", reactions.shape)
assert reports["safetyreportid"].is_unique, "Duplicate report IDs in reports table"
assert len(drugs) == reports["n_drugs"].sum(), "Drug row count mismatch"
assert len(reactions) == reports["n_reactions"].sum(), "Reaction row count mismatch"
print("Row-count checks passed.")

print("\nReports with 0 drugs:    ", (reports["n_drugs"] == 0).sum())
print("Reports with 0 reactions:", (reports["n_reactions"] == 0).sum())
print("\nDrugs per report (summary):\n", reports["n_drugs"].describe().round(1))
print("\nPatient sex codes:\n", reports["patient_sex"].value_counts(dropna=False))
print("\nAge unit codes:\n", reports["patient_age_unit"].value_counts(dropna=False))
print("\nDrug characterization codes:\n", drugs["drugcharacterization"].value_counts(dropna=False))
print("\nReaction outcome codes:\n", reactions["reactionoutcome"].value_counts(dropna=False))

Loaded records: 3100

Shapes -> reports: (3100, 20) | drugs: (11856, 8) | reactions: (9335, 4)
Row-count checks passed.

Reports with 0 drugs:     0
Reports with 0 reactions: 0

Drugs per report (summary):
 count    3100.0
mean        3.8
std         9.9
min         1.0
25%         1.0
50%         2.0
75%         3.0
max       259.0
Name: n_drugs, dtype: float64

Patient sex codes:
 patient_sex
2       1399
1       1260
None     441
Name: count, dtype: int64

Age unit codes:
 patient_age_unit
801     1779
None    1275
800       22
804       16
802        8
Name: count, dtype: int64

Drug characterization codes:
 drugcharacterization
1    7719
2    4094
3      43
Name: count, dtype: int64

Reaction outcome codes:
 reactionoutcome
6       5327
3       1340
1       1128
5        858
2        648
4         23
None      11
Name: count, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/openfda-project/data/raw
!mkdir -p /content/drive/MyDrive/openfda-project/data/processed
!cp data/raw/drug_event_sample_2024-01.json /content/drive/MyDrive/openfda-project/data/raw/
!cp data/processed/*_extracted.csv /content/drive/MyDrive/openfda-project/data/processed/
!ls -lh /content/drive/MyDrive/openfda-project/data/raw /content/drive/MyDrive/openfda-project/data/processed

Mounted at /content/drive
/content/drive/MyDrive/openfda-project/data/processed:
total 4.8M
-rw------- 1 root root 4.2M Oct 23 15:13 drugs_extracted.csv
-rw------- 1 root root 285K Oct 23 15:13 reactions_extracted.csv
-rw------- 1 root root 256K Oct 23 15:13 reports_extracted.csv

/content/drive/MyDrive/openfda-project/data/raw:
total 70M
-rw------- 1 root root 70M Oct 23 15:13 drug_event_sample_2024-01.json
